In [2]:
import pandas as pd

# Load FINAL dataset (do NOT modify original file)
df = pd.read_csv("Final_analysis_dataset_#4.csv")

# Quick check
print(df.columns)
print(len(df))

Index(['PMID', 'Title', 'Abstract', 'Publication Year', 'Authors',
       'Journal/Book', 'Black_mentions', 'South_Asian_mentions',
       'Asian_aggregation', 'Disaggregation'],
      dtype='object')
6652


In [3]:
black_df = df[df["Black_mentions"] == True].copy()

In [4]:
sa_df = df[df["South_Asian_mentions"] == True].copy()

In [5]:
black_df.to_csv("H2_black_base.csv", index=False)
sa_df.to_csv("H2_sa_base.csv", index=False)

print("H2 base datasets saved!")

H2 base datasets saved!


In [8]:
import pandas as pd

# -----------------------------
# 1. Load your FINAL dataset
# -----------------------------
df = pd.read_csv("Final_analysis_dataset_#4.csv")
df["Abstract"] = df["Abstract"].fillna("")

# -----------------------------
# 2. Create H2 subsets
# -----------------------------
black_df = df[df["Black_mentions"] == True].copy()
sa_df = df[df["South_Asian_mentions"] == True].copy()

print(f"Black subset: {len(black_df)} rows")
print(f"South Asian subset: {len(sa_df)} rows")

# -----------------------------
# 3. Add H2-specific columns
# -----------------------------
for subset in [black_df, sa_df]:
    subset["explicit_risk_flag"] = None
    subset["comparative_language"] = None
    subset["multi_site_or_national"] = None
    subset["landmark_candidate"] = None
    subset["citation_count"] = None
    subset["decade"] = (subset["Publication Year"] // 10) * 10

# -----------------------------
# 4. Save new CSVs (safe, no overwrite)
# -----------------------------
black_df.to_csv("H2_black_coding.csv", index=False)
sa_df.to_csv("H2_sa_coding.csv", index=False)

print("H2 coding CSVs created safely!")

Black subset: 2239 rows
South Asian subset: 1676 rows
H2 coding CSVs created safely!


In [9]:
import pandas as pd

# Load H2 CSVs
black_df = pd.read_csv("H2_black_coding.csv")
sa_df = pd.read_csv("H2_sa_coding.csv")

print(f"Black rows: {len(black_df)}, South Asian rows: {len(sa_df)}")

Black rows: 2239, South Asian rows: 1676


In [11]:
import pandas as pd

# -----------------------------
# 1. Load your H2 CSVs
# -----------------------------
black_df = pd.read_csv("H2_black_coding.csv")
sa_df = pd.read_csv("H2_sa_coding.csv")

for df in [black_df, sa_df]:
    df["Abstract"] = df["Abstract"].fillna("").str.lower()

# -----------------------------
# 2. Expanded risk keywords
# -----------------------------
risk_keywords = [
    # General elevated risk
"high risk", "elevated risk", "greater risk", "increased risk", "more likely", "excess risk",
"higher incidence", "higher prevalence", "higher rates", "increased incidence", "increased prevalence",
"greater incidence", "greater prevalence", "greater burden", "excess burden", "disproportionate burden",
"disproportionate risk", "higher morbidity", "higher mortality", "excess mortality",
"overrepresented", "underrepresented", "priority population", "high-risk population",
"disproportionate impact", "disproportionately affected", "unequal burden", "unequal risk",
"predisposed", "vulnerable population", "clinically at risk", "comparative risk",


# Explicit comparison phrases
"compared to whites", "compared with whites", "compared to non-hispanic whites",
"compared with non-hispanic whites", "relative risk", "risk ratio", "hazard ratio",
"odds ratio", "vs whites", "versus whites", "relative to whites",
"compared to other racial groups", "compared with other racial groups",
"higher than other groups", "greater than other groups", "versus", "vs"

]

# -----------------------------
# 3. Multi-site/national keywords
# -----------------------------
multi_keywords = [
    "nhanes", "framingham", "regards", "multi-center", "multi site", "multi-site",
    "national", "cohort", "population-based", "registry", "multi-hospital", "multi-institution"
]

# -----------------------------
# 4. Functions to auto-flag
# -----------------------------
def flag_explicit_risk(df):
    df['explicit_risk_flag_auto'] = df['Abstract'].apply(
        lambda x: any(k in x for k in risk_keywords)
    )
    return df

def flag_multi_site(df):
    df['multi_site_or_national_auto'] = df['Abstract'].apply(
        lambda x: any(k in x for k in multi_keywords)
    )
    return df

# -----------------------------
# 5. Apply flags
# -----------------------------
for df in [black_df, sa_df]:
    df = flag_explicit_risk(df)
    df = flag_multi_site(df)
    df['decade'] = (df['Publication Year'] // 10) * 10

# -----------------------------
# 6. Create review CSV (~50 rows per group)
# Filter to likely early landmark candidates
# -----------------------------
def make_review_csv(df, filename):
    review_df = df[
        (df['explicit_risk_flag_auto']) & 
        (df['multi_site_or_national_auto'])
    ].sort_values('Publication Year').head(50)
    review_df.to_csv(filename, index=False)
    print(f"{filename} saved with {len(review_df)} rows")

make_review_csv(black_df, "H2_black_review_candidates.csv")
make_review_csv(sa_df, "H2_sa_review_candidates.csv")

# -----------------------------
# 7. Save updated H2 CSVs with flags
# -----------------------------
black_df.to_csv("H2_black_coding_flagged.csv", index=False)
sa_df.to_csv("H2_sa_coding_flagged.csv", index=False)
print("Updated flagged CSVs saved.")

H2_black_review_candidates.csv saved with 50 rows
H2_sa_review_candidates.csv saved with 50 rows
Updated flagged CSVs saved.


In [13]:
import pandas as pd

def parse_pubmed_txt(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        record = {}
        for line in f:
            line = line.strip()
            if line.startswith("PMID-"):
                if record:
                    data.append(record)
                    record = {}
                record['PMID'] = line.replace("PMID-", "").strip()
            elif line.startswith("TI  -"):
                record['Title'] = line.replace("TI  -", "").strip()
            elif line.startswith("AB  -"):
                record['Abstract'] = line.replace("AB  -", "").strip()
            elif line.startswith("AU  -"):
                record['Authors'] = line.replace("AU  -", "").strip()
            elif line.startswith("DP  -"):
                # Only take the year part (first 4 digits)
                record['Publication Year'] = line.replace("DP  -", "").strip()[:4]
        if record:
            data.append(record)
    df = pd.DataFrame(data)
    # Fill missing abstracts with empty strings
    df['Abstract'] = df['Abstract'].fillna("").str.lower()
    return df

# Parse Black Americans
black_df = parse_pubmed_txt("H2_black.txt")

# Parse South Asians
sa_df = parse_pubmed_txt("H2_sa.txt")

print(f"Black: {len(black_df)} records, South Asian: {len(sa_df)} records")

Black: 3499 records, South Asian: 2218 records


In [14]:
risk_keywords = [
   # General elevated risk
"high risk", "elevated risk", "greater risk", "increased risk", "more likely", "excess risk",
"higher incidence", "higher prevalence", "higher rates", "increased incidence", "increased prevalence",
"greater incidence", "greater prevalence", "greater burden", "excess burden", "disproportionate burden",
"disproportionate risk", "higher morbidity", "higher mortality", "excess mortality",
"overrepresented", "underrepresented", "priority population", "high-risk population",
"disproportionate impact", "disproportionately affected", "unequal burden", "unequal risk",
"predisposed", "vulnerable population", "clinically at risk", "comparative risk",


# Explicit comparison phrases
"compared to whites", "compared with whites", "compared to non-hispanic whites",
"compared with non-hispanic whites", "relative risk", "risk ratio", "hazard ratio",
"odds ratio", "vs whites", "versus whites", "relative to whites",
"compared to other racial groups", "compared with other racial groups",
"higher than other groups", "greater than other groups"

]

black_df['explicit_risk'] = black_df['Abstract'].apply(lambda x: any(k in x for k in risk_keywords))
sa_df['explicit_risk'] = sa_df['Abstract'].apply(lambda x: any(k in x for k in risk_keywords))

In [15]:
multi_keywords = [
    "multi-center", "multi site", "multi-site", "national", "nhanes", "framingham",
    "cohort", "population-based", "registry", "multi-hospital", "multi-institution"
]

black_df['multi_site'] = black_df['Abstract'].apply(lambda x: any(k in x for k in multi_keywords))
sa_df['multi_site'] = sa_df['Abstract'].apply(lambda x: any(k in x for k in multi_keywords))

In [16]:
epi_keywords = [
    # Study design
    "cohort", "case-control", "cross-sectional", "ecologic",
    "randomized controlled trial", "rct", "longitudinal", "follow-up",
    "nested case-control", "surveillance",
    # Disease frequency & distribution
    "incidence", "prevalence", "attack rate", "mortality rate", "morbidity",
    "epidemic", "pandemic", "endemic", "outbreak", "cluster",
    # Measures of association & risk
    "relative risk", "odds ratio", "risk factor", "protective factor",
    "confidence interval", "p-value", "hazard ratio",
    # Concepts & analysis
    "exposure", "determinants", "population-based", "confounding",
    "bias", "effect modification", "surveillance", "transmission", "epidemiology"
]

black_df['epi_flag'] = black_df['Abstract'].apply(lambda x: any(k in x for k in epi_keywords))
sa_df['epi_flag'] = sa_df['Abstract'].apply(lambda x: any(k in x for k in epi_keywords))

In [17]:
black_df.sort_values('Publication Year', inplace=True)
sa_df.sort_values('Publication Year', inplace=True)

black_df.to_csv("H2_black_review_candidates.csv", index=False)
sa_df.to_csv("H2_sa_review_candidates.csv", index=False)

In [19]:
import pandas as pd

# -----------------------------
# 1. Load your SA review CSV
# -----------------------------
sa_df = pd.read_csv("H2_sa_review_candidates.csv")  # replace with your file
sa_df["Abstract"] = sa_df["Abstract"].fillna("")
sa_df["Affiliation"] = sa_df.get("Affiliation", "").fillna("")  # use if exported

# -----------------------------
# 2. Flag explicit risk
# -----------------------------
explicit_risk_keywords = [
    "elevated cardiovascular risk", "high-risk population",
    "disproportionate risk", "increased risk", "higher risk",
    "higher incidence", "higher prevalence", "greater likelihood",
    "more likely to develop", "predisposed to", "susceptible to"
]

def flag_explicit_risk(text):
    text_lower = str(text).lower()
    return any(kw.lower() in text_lower for kw in explicit_risk_keywords)

sa_df["Explicit_risk"] = sa_df["Abstract"].apply(flag_explicit_risk)

# -----------------------------
# 3. Flag US-based studies
# -----------------------------
us_keywords = [
    "united states", "usa", "u.s.", "u.s.a", 
    "harvard", "penn", "nih", "johns hopkins", "columbia university", 
    "boston", "new york", "philadelphia", "chicago", "los angeles"
    # add other known US institutions if needed
]

def flag_us_based(abstract, affiliation):
    abstract_text = str(abstract).lower()
    affiliation_text = str(affiliation).lower()
    if any(kw in affiliation_text for kw in us_keywords):
        return True
    if any(kw in abstract_text for kw in us_keywords):
        return True
    return False

sa_df["US_based"] = sa_df.apply(lambda row: flag_us_based(row["Abstract"], row["Affiliation"]), axis=1)

# -----------------------------
# 4. Filter to ready-to-review candidates
# -----------------------------
ready_df = sa_df[(sa_df["Explicit_risk"]) & (sa_df["US_based"])]

# Optional: sort by publication year ascending
ready_df = ready_df.sort_values(by="Publication Year")

# -----------------------------
# 5. Save to new CSV for manual review
# -----------------------------
ready_df.to_csv("SA_review_H2_#2.csv", index=False)

print(f"Number of SA studies flagged for review: {len(ready_df)}")

AttributeError: 'str' object has no attribute 'fillna'

In [20]:
import pandas as pd

# -----------------------------
# 1. Load your SA review CSV
# -----------------------------
sa_df = pd.read_csv("H2_sa_review_candidates.csv")  # replace with your file

# Ensure the columns exist and are proper Pandas Series
if "Abstract" not in sa_df.columns:
    sa_df["Abstract"] = ""  # create empty column if missing
if "Affiliation" not in sa_df.columns:
    sa_df["Affiliation"] = ""  # create empty column if missing

# Fill missing values with empty strings
sa_df["Abstract"] = sa_df["Abstract"].astype(str).fillna("")
sa_df["Affiliation"] = sa_df["Affiliation"].astype(str).fillna("")

# -----------------------------
# 2. Flag explicit risk
# -----------------------------
explicit_risk_keywords = [
    "elevated cardiovascular risk", "high-risk population",
    "disproportionate risk", "increased risk", "higher risk",
    "higher incidence", "higher prevalence", "greater likelihood",
    "more likely to develop", "predisposed to", "susceptible to"
]

def flag_explicit_risk(text):
    text_lower = str(text).lower()
    return any(kw.lower() in text_lower for kw in explicit_risk_keywords)

sa_df["Explicit_risk"] = sa_df["Abstract"].apply(flag_explicit_risk)

# -----------------------------
# 3. Flag US-based studies
# -----------------------------
us_keywords = [
    "united states", "usa", "u.s.", "u.s.a",
    "harvard", "penn", "nih", "johns hopkins", "columbia university",
    "boston", "new york", "philadelphia", "chicago", "los angeles"
]

def flag_us_based(abstract, affiliation):
    abstract_text = str(abstract).lower()
    affiliation_text = str(affiliation).lower()
    if any(kw in affiliation_text for kw in us_keywords):
        return True
    if any(kw in abstract_text for kw in us_keywords):
        return True
    return False

sa_df["US_based"] = sa_df.apply(lambda row: flag_us_based(row["Abstract"], row["Affiliation"]), axis=1)

# -----------------------------
# 4. Filter to ready-to-review candidates
# -----------------------------
ready_df = sa_df[(sa_df["Explicit_risk"]) & (sa_df["US_based"])]

# Optional: sort by publication year ascending
if "Publication Year" in ready_df.columns:
    ready_df = ready_df.sort_values(by="Publication Year")

# -----------------------------
# 5. Save to new CSV for manual review
# -----------------------------
ready_df.to_csv("SA_review_H2_#2.csv", index=False)

print(f"Number of SA studies flagged for review: {len(ready_df)}")

Number of SA studies flagged for review: 0


In [21]:
print(sa_df.head()[["Title", "Abstract", "Affiliation"]])

                                               Title  \
0  Prevalence and reproducibility of exercise-ind...   
1    The prevalence of hypertension in urban whites.   
2  Rheumatic fever and rheumatic heart disease in...   
3  Women at altitude: cardiovascular responses to...   
4  Infective endocarditis in thirteen children: a...   

                                            Abstract Affiliation  
0  the occurrence of ventricular arrhythmias at r...              
1  in a random house-to-house study of 1006 white...              
2  a preliminary pilot study of streptococcal inf...              
3  six women mountaineers, 23-43 years of age, pa...              
4  thirteen children (11 african, two indian) wit...              


In [22]:
for kw in explicit_risk_keywords:
    matches = sa_df[sa_df["Abstract"].str.contains(kw, case=False, na=False)]
    print(f"{kw}: {len(matches)} matches")

elevated cardiovascular risk: 0 matches
high-risk population: 0 matches
disproportionate risk: 0 matches
increased risk: 25 matches
higher risk: 17 matches
higher incidence: 3 matches
higher prevalence: 8 matches
greater likelihood: 0 matches
more likely to develop: 0 matches
predisposed to: 7 matches
susceptible to: 4 matches


In [23]:
explicit_risk_keywords = [
    "increased risk",       # matched 25 times
    "higher risk",          # matched 17 times
    "higher incidence",     # matched 3 times
    "higher prevalence",    # matched 8 times
    "predisposed to",       # matched 7 times
    "susceptible to",       # matched 4 times
    "more likely to develop",
    "greater likelihood",
    "risk factor",          # new: very common phrasing
    "risk of heart disease", 
    "risk of cardiovascular disease",
    "risk for coronary heart disease",
    "higher rates of heart disease",
    "increased incidence of cardiovascular disease",
    "more likely to experience cardiovascular disease"
]

In [24]:
def flag_explicit_risk(text):
    text_lower = str(text).lower()
    return any(kw.lower() in text_lower for kw in explicit_risk_keywords)

sa_df["Explicit_risk"] = sa_df["Abstract"].apply(flag_explicit_risk)

In [25]:
ready_df = sa_df[sa_df["Explicit_risk"]]
ready_df = ready_df.sort_values(by="Publication Year")
ready_df.to_csv("SA_review_ready_for_H2_shortlist.csv", index=False)
print(f"Number of SA studies flagged for review: {len(ready_df)}")

Number of SA studies flagged for review: 182


In [28]:
import pandas as pd

# Load the original text file
with open("H2_SA.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split into individual records by 'PMID-'
records = raw_text.split("\nPMID- ")

data = []
for rec in records:
    lines = rec.split("\n")
    pmid = lines[0].strip() if lines[0].strip().isdigit() else None
    title = ""
    abstract = ""
    affiliation = ""
    year = None
    for line in lines:
        if line.startswith("TI -"):
            title = line.replace("TI -", "").strip()
        elif line.startswith("AB -"):
            abstract += line.replace("AB -", "").strip() + " "
        elif line.startswith("AD -"):  # affiliation
            affiliation = line.replace("AD -", "").strip()
        elif line.startswith("DP -"):  # publication date
            year = line.replace("DP -", "").strip()[:4]  # just the year
    if pmid:
        data.append([pmid, title, abstract.strip(), affiliation, year])

# Create DataFrame
sa_df = pd.DataFrame(data, columns=["PMID", "Title", "Abstract", "Affiliation", "Publication Year"])
print(f"Loaded {len(sa_df)} records from H2_SA.txt")

Loaded 2217 records from H2_SA.txt


In [29]:
explicit_risk_keywords = [
    "increased risk", "higher risk", "higher incidence", "higher prevalence",
    "predisposed to", "susceptible to", "more likely to develop", "greater likelihood",
    "risk factor", "risk of heart disease", "risk of cardiovascular disease",
    "risk for coronary heart disease", "higher rates of heart disease",
    "increased incidence of cardiovascular disease", "more likely to experience cardiovascular disease"
]

def flag_explicit_risk(text):
    text_lower = str(text).lower()
    return any(kw.lower() in text_lower for kw in explicit_risk_keywords)

sa_df["Explicit_risk"] = sa_df["Abstract"].apply(flag_explicit_risk)

In [30]:
epi_design_keywords = [
    "cohort", "case-control", "cross-sectional", "ecologic", 
    "randomized controlled trial", "longitudinal", "follow-up", 
    "nested case-control", "surveillance", "population-based"
]

def flag_epi_design(text):
    text_lower = str(text).lower()
    return any(kw.lower() in text_lower for kw in epi_design_keywords)

sa_df["Epi_design"] = sa_df["Abstract"].apply(flag_epi_design)

In [31]:
ready_df = sa_df[(sa_df["Explicit_risk"]) | (sa_df["Epi_design"])]
ready_df = ready_df.sort_values(by="Publication Year")  # earliest first

ready_df.to_csv("SA_review_ready_for_H2_manual.csv", index=False)
print(f"Number of SA studies flagged for review: {len(ready_df)}")

Number of SA studies flagged for review: 0


In [32]:
# Loosen filter: keep all studies, or just those with any abstract text
ready_df = sa_df[sa_df["Abstract"].str.strip() != ""]
ready_df = ready_df.sort_values(by="Publication Year")

# Save to CSV for manual review
ready_df.to_csv("SA_review_ready_for_H2_manual.csv", index=False)
print(f"Number of SA studies flagged for review (loose filter): {len(ready_df)}")

Number of SA studies flagged for review (loose filter): 0


In [33]:
import pandas as pd

# Load the raw text
with open("H2_SA.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split records by 'PMID-'
records = raw_text.split("\nPMID- ")
data = []

for rec in records:
    lines = rec.split("\n")
    pmid = lines[0].strip() if lines[0].strip().isdigit() else None
    title, abstract, affiliation, year = "", "", "", None
    capture_ab = False  # flag to capture multi-line abstracts

    for line in lines[1:]:
        if line.startswith("TI -"):
            title = line.replace("TI -", "").strip()
        elif line.startswith("AB -"):
            abstract = line.replace("AB -", "").strip()
            capture_ab = True
        elif line.startswith("AD -"):
            affiliation = line.replace("AD -", "").strip()
        elif line.startswith("DP -"):
            year = line.replace("DP -", "").strip()[:4]
        elif capture_ab:
            # Continue capturing multi-line abstract until next field
            if line.startswith(("TI -", "AD -", "DP -", "FAU -", "AU -", "MH -")):
                capture_ab = False
            else:
                abstract += " " + line.strip()

    if pmid:
        data.append([pmid, title, abstract.strip(), affiliation.strip(), year])

# Create DataFrame
sa_df = pd.DataFrame(data, columns=["PMID", "Title", "Abstract", "Affiliation", "Publication Year"])

# Quick check
print(f"Loaded {len(sa_df)} records")
print(sa_df[["PMID", "Title", "Abstract"]].head())

Loaded 2217 records
      PMID Title Abstract
0  7404085               
1  7032733               
2  7092756               
3  6185079               
4  7089765               


In [34]:
ready_df = sa_df[sa_df["Abstract"].str.strip() != ""]
ready_df = ready_df.sort_values(by="Publication Year")
ready_df.to_csv("SA_review_ready_for_H2_manual.csv", index=False)
print(f"Number of SA studies ready for manual review: {len(ready_df)}")

Number of SA studies ready for manual review: 0


In [35]:
with open("H2_SA.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(lines[:50])  # first 50 lines of the file

['PMID- 1258800\n', 'OWN - NLM\n', 'STAT- MEDLINE\n', 'DCOM- 19760602\n', 'LR  - 20190622\n', 'IS  - 0002-9149 (Print)\n', 'IS  - 0002-9149 (Linking)\n', 'VI  - 37\n', 'IP  - 4\n', 'DP  - 1976 Mar 31\n', 'TI  - Prevalence and reproducibility of exercise-induced ventricular arrhythmias during \n', '      maximal exercise testing in normal men.\n', 'PG  - 617-22\n', 'AB  - The occurrence of ventricular arrhythmias at rest or during ordinary daily \n', '      activities has been implicated as a risk factor for future coronary-related \n', '      events and sudden death. However, the clerical significance of exercise-induced \n', '      ventricular arrhythmias remains uncertain. To assess the prevalence and \n', '      reproducibility of such arrhythmias, two serial maximal treadmill exercise tests \n', '      were performed in a study population of 543 male Indian State policemen at an \n', '      average interval of 2.9 years. Four hundred sixty-two subjects were clinically \n', '      f

In [36]:
import pandas as pd

# Load the raw text
with open("H2_SA.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

data = []

pmid = None
title = ""
abstract = ""
affiliation = ""
year = None
capture_ab = False  # flag to capture multi-line abstract

for line in lines:
    line = line.rstrip()  # remove trailing newline

    if line.startswith("PMID-"):
        # Save previous record if exists
        if pmid:
            data.append([pmid, title, abstract.strip(), affiliation, year])
        # Start new record
        pmid = line.replace("PMID-", "").strip()
        title = ""
        abstract = ""
        affiliation = ""
        year = None
        capture_ab = False

    elif line.startswith("TI -"):
        title = line.replace("TI -", "").strip()
    elif line.startswith("AB -"):
        abstract = line.replace("AB -", "").strip()
        capture_ab = True
    elif line.startswith("AD -"):
        affiliation = line.replace("AD -", "").strip()
    elif line.startswith("DP -"):
        year = line.replace("DP -", "").strip()[:4]
    elif capture_ab:
        # If line starts with another field, stop capturing abstract
        if any(line.startswith(f) for f in ["TI -","AD -","DP -","FAU -","AU -","MH -","PT -"]):
            capture_ab = False
        else:
            abstract += " " + line.strip()

# Add the last record
if pmid:
    data.append([pmid, title, abstract.strip(), affiliation, year])

# Create DataFrame
sa_df = pd.DataFrame(data, columns=["PMID","Title","Abstract","Affiliation","Publication Year"])

# Quick check
print(f"Loaded {len(sa_df)} records")
print(sa_df[["PMID","Title","Abstract"]].head())

Loaded 2218 records
      PMID Title Abstract
0  1258800               
1  7404085               
2  7032733               
3  7092756               
4  6185079               


In [37]:
import pandas as pd

# Load raw text
with open("H2_SA.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

data = []

pmid = ""
title = ""
abstract = ""
affiliation = ""
year = None
capture_ab = False

for line in lines:
    line = line.rstrip()

    if line.startswith("PMID-"):
        if pmid:
            data.append([pmid, title, abstract.strip(), affiliation, year])
        pmid = line.replace("PMID-", "").strip()
        title = ""
        abstract = ""
        affiliation = ""
        year = None
        capture_ab = False

    elif line.startswith("TI -"):
        title = line.replace("TI -", "").strip()

    elif line.startswith("AB -"):
        abstract = line.replace("AB -", "").strip()
        capture_ab = True

    elif line.startswith("AD -"):
        affiliation = line.replace("AD -", "").strip()

    elif line.startswith("DP -"):
        year = line.replace("DP -", "").strip()[:4]

    elif capture_ab:
        # Continue capturing abstract until another field starts
        if any(line.startswith(f) for f in ["TI -", "AD -", "DP -", "FAU -", "AU -", "MH -", "PT -"]):
            capture_ab = False
        else:
            abstract += " " + line.strip()

# Add last record
if pmid:
    data.append([pmid, title, abstract.strip(), affiliation, year])

# Create DataFrame
sa_df = pd.DataFrame(data, columns=["PMID","Title","Abstract","Affiliation","Publication Year"])

# Quick check
print(f"Loaded {len(sa_df)} records")
print(sa_df[["PMID","Title","Abstract"]].head())

Loaded 2218 records
      PMID Title Abstract
0  1258800               
1  7404085               
2  7032733               
3  7092756               
4  6185079               


In [38]:
import pandas as pd

# Path to your H2 SA text file
file_path = "H2_SA.txt"

# Open the file and read lines
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Initialize storage for records
records = []
current_record = {}

for line in lines:
    line = line.strip()
    
    if not line:
        # Blank line indicates end of a record
        if current_record:
            records.append(current_record)
            current_record = {}
        continue
    
    # Extract fields
    if line.startswith("PMID-") or line.startswith("PMID:"):
        current_record["PMID"] = line.replace("PMID-", "").replace("PMID:", "").strip()
    elif line.startswith("TI -") or line.startswith("Title:"):
        current_record["Title"] = line.replace("TI -", "").replace("Title:", "").strip()
    elif line.startswith("AB -") or line.startswith("Abstract:"):
        current_record["Abstract"] = line.replace("AB -", "").replace("Abstract:", "").strip()
    elif line.startswith("AD -") or line.startswith("Affiliation:"):
        current_record["Affiliation"] = line.replace("AD -", "").replace("Affiliation:", "").strip()
    elif line.startswith("DP -") or line.startswith("Publication Year:"):
        current_record["Publication Year"] = line.replace("DP -", "").replace("Publication Year:", "").strip()[:4]
    else:
        # For multi-line abstracts, append
        if "Abstract" in current_record:
            current_record["Abstract"] += " " + line

# Catch last record if no blank line at the end
if current_record:
    records.append(current_record)

# Convert to DataFrame
df = pd.DataFrame(records)

# Preview DataFrame
print(f"Loaded {len(df)} records")
print(df.head())

# Save to CSV
df.to_csv("H2_SA_abstracts_parsed.csv", index=False)

Loaded 2218 records
      PMID
0  1258800
1  7404085
2  7032733
3  7092756
4  6185079


In [39]:
import pandas as pd

file_path = "H2_SA.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

records = []
current = {}
capture_abstract = False

for line in lines:
    line = line.rstrip()  # remove trailing newline and spaces
    
    if line.startswith("PMID-"):
        # Save previous record
        if current:
            records.append(current)
        # Start new record
        current = {"PMID": line.replace("PMID-", "").strip(),
                   "Title": "", "Abstract": "", "Affiliation": "", "Publication Year": ""}
        capture_abstract = False

    elif line.startswith("TI -"):
        current["Title"] = line.replace("TI -", "").strip()

    elif line.startswith("AB -"):
        current["Abstract"] = line.replace("AB -", "").strip()
        capture_abstract = True

    elif line.startswith("AD -"):
        current["Affiliation"] = line.replace("AD -", "").strip()

    elif line.startswith("DP -"):
        current["Publication Year"] = line.replace("DP -", "").strip()[:4]

    else:
        # Multi-line abstract continuation
        if capture_abstract:
            # Stop capturing if new field starts
            if any(line.startswith(f) for f in ["TI -","AD -","DP -","FAU -","AU -","MH -","PT -"]):
                capture_abstract = False
            else:
                current["Abstract"] += " " + line.strip()

# Append last record
if current:
    records.append(current)

# Convert to DataFrame
df = pd.DataFrame(records)

# Filter out records with empty abstracts
df = df[df["Abstract"].str.strip() != ""]

# Sort by Publication Year for earliest studies
df = df.sort_values("Publication Year")

# Save CSV
df.to_csv("H2_SA_abstracts_parsed.csv", index=False)

# Quick check
print(f"Loaded {len(df)} records")
print(df.head())

Loaded 0 records
Empty DataFrame
Columns: [PMID, Title, Abstract, Affiliation, Publication Year]
Index: []


In [40]:
import pandas as pd
import re

file_path = "H2_SA.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

records = []
current = {}
capture_abstract = False

for line in lines:
    line = line.rstrip()

    if re.match(r'^PMID- ?', line):
        if current:
            records.append(current)
        current = {"PMID": re.sub(r'^PMID- ?', '', line).strip(),
                   "Title": "", "Abstract": "", "Affiliation": "", "Publication Year": ""}
        capture_abstract = False

    elif re.match(r'^TI\s*-\s*', line):
        current["Title"] = re.sub(r'^TI\s*-\s*', '', line).strip()

    elif re.match(r'^AB\s*-\s*', line):
        current["Abstract"] = re.sub(r'^AB\s*-\s*', '', line).strip()
        capture_abstract = True

    elif re.match(r'^AD\s*-\s*', line):
        current["Affiliation"] = re.sub(r'^AD\s*-\s*', '', line).strip()

    elif re.match(r'^DP\s*-\s*', line):
        current["Publication Year"] = re.sub(r'^DP\s*-\s*', '', line).strip()[:4]

    else:
        # multi-line abstract continuation
        if capture_abstract:
            if re.match(r'^(TI|AB|AD|DP|FAU|AU|MH|PT)\s*-', line):
                capture_abstract = False
            else:
                current["Abstract"] += " " + line.strip()

# append last record
if current:
    records.append(current)

df = pd.DataFrame(records)

# Remove empty abstracts
df = df[df["Abstract"].str.strip() != ""]

# Sort by Publication Year
df = df.sort_values("Publication Year")

# Save CSV
df.to_csv("H2_SA_abstracts_parsed.csv", index=False)

print(f"Loaded {len(df)} records")
print(df.head())

Loaded 2181 records
      PMID                                              Title  \
0  1258800  Prevalence and reproducibility of exercise-ind...   
1  7404085    The prevalence of hypertension in urban whites.   
2  7032733  Rheumatic fever and rheumatic heart disease in...   
3  7092756  Women at altitude: cardiovascular responses to...   
4  6185079  Infective endocarditis in thirteen children: a...   

                                            Abstract Affiliation  \
0  The occurrence of ventricular arrhythmias at r...               
1  In a random house-to-house study of 1006 White...               
2  A preliminary pilot study of streptococcal inf...               
3  Six women mountaineers, 23-43 years of age, pa...               
4  Thirteen children (11 African, two Indian) wit...               

  Publication Year  
0             1976  
1             1980  
2             1981  
3             1982  
4             1982  


In [41]:
risk_kw = ["increased risk", "higher risk", "predisposed to", "susceptible to", 
           "more likely to develop", "higher incidence", "higher prevalence", "high-risk population", "disproportionate risk"]
epi_kw = ["cohort", "case-control", "cross-sectional", "longitudinal", "population-based", "follow-up", "surveillance", "incidence", "prevalence", "risk factor", "determinants"]
us_kw = ["United States", "multi-center", "multi-site", "national", "CDC", "NHANES", "community health centers"]

df["Explicit_Risk"] = df["Abstract"].str.lower().apply(lambda x: any(k in x for k in risk_kw))
df["Epidemiological"] = df["Abstract"].str.lower().apply(lambda x: any(k in x for k in epi_kw))
df["US_or_Multi"] = df["Abstract"].str.lower().apply(lambda x: any(k.lower() in x for k in us_kw))

In [42]:
review_candidates = df[df["Explicit_Risk"] & df["Epidemiological"] & df["US_or_Multi"]]
review_candidates.to_csv("H2_SA_review_candidates.csv", index=False)
print(f"Number of SA studies flagged for review: {len(review_candidates)}")

Number of SA studies flagged for review: 67


In [47]:
# ===============================================
# Python Script: Parse PubMed Exports for Black and South Asian Studies
# Handles CSV metadata + TXT abstracts
# ===============================================

import pandas as pd

# -----------------------------
# Step 1: Define file paths
# -----------------------------
# Replace these with your actual file names
black_meta_csv = "H2_black.csv"          # metadata CSV
black_abstract_txt = "H2_black_abstracts.txt" # abstract TXT

sa_meta_csv = "H2_SA.csv"                # metadata CSV
sa_abstract_txt = "H2_SA_abstracts.txt"       # abstract TXT

# -----------------------------
# Step 2: Function to parse abstract TXT
# -----------------------------
def parse_pubmed_abstract_txt(file_path):
    pmid_list = []
    abstract_list = []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    pmid = abstract = ""
    for line in lines:
        line = line.strip()
        
        if line.startswith("PMID-"):
            if pmid:  # save previous record
                pmid_list.append(pmid)
                abstract_list.append(abstract)
                abstract = ""
            pmid = line.replace("PMID- ", "")
        
        elif line.startswith("AB  -"):
            abstract = line.replace("AB  - ", "")
        else:
            # multi-line abstracts
            if abstract:
                abstract += " " + line

    # Add last record
    if pmid:
        pmid_list.append(pmid)
        abstract_list.append(abstract)
    
    return pd.DataFrame({"PMID": pmid_list, "Abstract": abstract_list})

# -----------------------------
# Step 3: Load metadata CSVs
# -----------------------------
df_black_meta = pd.read_csv(black_meta_csv, encoding="utf-8")
df_sa_meta = pd.read_csv(sa_meta_csv, encoding="utf-8")

# -----------------------------
# Step 4: Parse abstracts TXT
# -----------------------------
df_black_abs = parse_pubmed_abstract_txt(black_abstract_txt)
df_sa_abs = parse_pubmed_abstract_txt(sa_abstract_txt)

# -----------------------------
# Step 5: Merge metadata + abstracts
# -----------------------------
df_black = pd.merge(df_black_meta, df_black_abs, on="PMID", how="inner")
df_sa = pd.merge(df_sa_meta, df_sa_abs, on="PMID", how="inner")

In [48]:
us_keywords = [
    "United States", "USA", "U.S.", "US", "U.S.A.", "United States of America"
]

In [49]:
df_black_us = df_black[df_black['Affiliation'].str.contains('|'.join(us_keywords), case=False, na=False)]
df_sa_us = df_sa[df_sa['Affiliation'].str.contains('|'.join(us_keywords), case=False, na=False)]

KeyError: 'Affiliation'

In [50]:
print(df_black.columns)
print(df_sa.columns)

Index(['PMID', 'Title', 'Authors', 'Citation', 'First Author', 'Journal/Book',
       'Publication Year', 'Create Date', 'PMCID', 'NIHMS ID', 'DOI',
       'Abstract'],
      dtype='object')
Index(['PMID', 'Title', 'Authors', 'Citation', 'First Author', 'Journal/Book',
       'Publication Year', 'Create Date', 'PMCID', 'NIHMS ID', 'DOI',
       'Abstract'],
      dtype='object')


In [51]:
# -----------------------------
# Step 2: Define expanded risk keywords
# -----------------------------
risk_keywords = [
    "disproportionate",
    "higher risk",
    "elevated risk",
    "increased risk",
    "greater risk",
    "excess risk",
    "predisposition",
    "vulnerable population",
    "high-risk population",
    "priority population",
    "disparities",
    "ethnic differences",
    "racial differences",
    "unequal burden",
    "excess burden",
    "higher prevalence",
    "greater incidence",
]

# -----------------------------
# Step 3: Flag abstracts
# -----------------------------
df_black['flag_risk'] = df_black['Abstract'].str.contains('|'.join(risk_keywords), case=False, na=False)
df_sa['flag_risk'] = df_sa['Abstract'].str.contains('|'.join(risk_keywords), case=False, na=False)

df_black_risk = df_black[df_black['flag_risk']]
df_sa_risk = df_sa[df_sa['flag_risk']]

print(f"Black studies flagged for disproportionate risk: {len(df_black_risk)}")
print(f"South Asian studies flagged for disproportionate risk: {len(df_sa_risk)}")

# -----------------------------
# Step 4: Save cleaned CSVs
# -----------------------------
df_black_risk.to_csv("black_disproportionate.csv", index=False)
df_sa_risk.to_csv("sa_disproportionate.csv", index=False)

print("CSV files saved: 'black_disproportionate.csv' and 'sa_disproportionate.csv'")

AttributeError: Can only use .str accessor with string values!

In [52]:
import pandas as pd
import os

# -----------------------------
# Step 1: Define file paths
# -----------------------------
# Replace with your actual file names
black_meta_csv = "H2_black.csv"          # Metadata CSV
black_abstract_txt = "H2_black_abstracts.txt" # Abstract TXT

sa_meta_csv = "H2_SA.csv"                # Metadata CSV
sa_abstract_txt = "H2_SA_abstracts.txt"       # Abstract TXT

In [53]:
# -----------------------------
# Step 2: Function to parse PubMed TXT abstracts
# -----------------------------
def parse_pubmed_abstract_txt(file_path):
    """
    Parse PubMed TXT export into a DataFrame with PMID + Abstract
    Handles multi-line abstracts
    """
    pmid_list = []
    abstract_list = []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    pmid = ""
    abstract = ""
    for line in lines:
        line = line.strip()
        
        if line.startswith("PMID-"):
            if pmid:  # Save previous record
                pmid_list.append(pmid)
                abstract_list.append(abstract)
                abstract = ""
            pmid = line.replace("PMID- ", "")
        
        elif line.startswith("AB  -"):
            abstract = line.replace("AB  - ", "")
        else:
            # Multi-line abstracts
            if abstract:
                abstract += " " + line

    # Add last record
    if pmid:
        pmid_list.append(pmid)
        abstract_list.append(abstract)
    
    return pd.DataFrame({"PMID": pmid_list, "Abstract": abstract_list})

In [54]:
# -----------------------------
# Step 3: Load metadata CSVs
# -----------------------------
df_black_meta = pd.read_csv(black_meta_csv, encoding="utf-8")
df_sa_meta = pd.read_csv(sa_meta_csv, encoding="utf-8")

# -----------------------------
# Step 4: Parse abstracts TXT
# -----------------------------
df_black_abs = parse_pubmed_abstract_txt(black_abstract_txt)
df_sa_abs = parse_pubmed_abstract_txt(sa_abstract_txt)

# -----------------------------
# Step 5: Merge metadata + abstracts on PMID
# -----------------------------
df_black = pd.merge(df_black_meta, df_black_abs, on="PMID", how="inner")
df_sa = pd.merge(df_sa_meta, df_sa_abs, on="PMID", how="inner")

# -----------------------------
# Step 6: Convert Abstracts to string (handle missing values)
# -----------------------------
df_black['Abstract'] = df_black['Abstract'].astype(str)
df_sa['Abstract'] = df_sa['Abstract'].astype(str)


In [55]:
# -----------------------------
# Step 7: Define expanded risk keywords
# -----------------------------
risk_keywords = [
    "disproportionate",
    "higher risk",
    "elevated risk",
    "increased risk",
    "greater risk",
    "excess risk",
    "predisposition",
    "vulnerable population",
    "high-risk population",
    "priority population",
    "disparities",
    "ethnic differences",
    "racial differences",
    "unequal burden",
    "excess burden",
    "higher prevalence",
    "greater incidence",
]


In [56]:
# -----------------------------
# Step 8: Flag abstracts mentioning disproportionate risk
# -----------------------------
df_black['flag_risk'] = df_black['Abstract'].str.contains('|'.join(risk_keywords), case=False, na=False)
df_sa['flag_risk'] = df_sa['Abstract'].str.contains('|'.join(risk_keywords), case=False, na=False)

df_black_risk = df_black[df_black['flag_risk']].copy()
df_sa_risk = df_sa[df_sa['flag_risk']].copy()

print(f"Black studies flagged for disproportionate risk: {len(df_black_risk)}")
print(f"South Asian studies flagged for disproportionate risk: {len(df_sa_risk)}")

# -----------------------------
# Step 9: Save cleaned CSVs
# -----------------------------
df_black_risk.to_csv("black_disproportionate.csv", index=False)
df_sa_risk.to_csv("sa_disproportionate.csv", index=False)

print("CSV files saved in folder:", os.getcwd())

Black studies flagged for disproportionate risk: 0
South Asian studies flagged for disproportionate risk: 0
CSV files saved in folder: /Users/mehtaabrakkar/anaconda_projects/d423a35d-1680-4d5b-8c4d-54db7ecbbe17


In [57]:
# Print first 5 abstracts to see actual text
print(df_black['Abstract'].head())
print(df_sa['Abstract'].head())

Series([], Name: Abstract, dtype: object)
Series([], Name: Abstract, dtype: object)


In [58]:
def parse_pubmed_abstract_txt(file_path):
    pmid_list = []
    abstract_list = []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    pmid = ""
    abstract = ""
    for line in lines:
        line = line.strip()
        
        if line.startswith("PMID-"):
            if pmid:  # save previous
                pmid_list.append(pmid)
                abstract_list.append(abstract)
                abstract = ""
            pmid = line.replace("PMID- ", "")
        elif line.startswith("AB  -"):
            abstract = line.replace("AB  - ", "")
        else:
            if abstract:
                abstract += " " + line
    if pmid:
        pmid_list.append(pmid)
        abstract_list.append(abstract)
    
    return pd.DataFrame({"PMID": pmid_list, "Abstract": abstract_list})

In [60]:
df_black_abs = parse_pubmed_abstract_txt("H2_black_abstracts.txt")
df_black = pd.merge(df_black_meta, df_black_abs, on="PMID", how="inner")

df_sa_abs = parse_pubmed_abstract_txt("H2_SA_abstracts.txt")
df_sa = pd.merge(df_sa_meta, df_sa_abs, on="PMID", how="inner")

In [61]:
# Merge CSV metadata + TXT abstracts on PMID
df_black = pd.merge(df_black_meta, df_black_abs, on="PMID", how="inner")
df_sa = pd.merge(df_sa_meta, df_sa_abs, on="PMID", how="inner")

# Quick check
print(df_black.head())
print(df_sa.head())

Empty DataFrame
Columns: [PMID, Title, Authors, Citation, First Author, Journal/Book, Publication Year, Create Date, PMCID, NIHMS ID, DOI, Abstract]
Index: []
Empty DataFrame
Columns: [PMID, Title, Authors, Citation, First Author, Journal/Book, Publication Year, Create Date, PMCID, NIHMS ID, DOI, Abstract]
Index: []


In [62]:
# Make sure PMIDs are strings and strip whitespace
df_black_meta['PMID'] = df_black_meta['PMID'].astype(str).str.strip()
df_sa_meta['PMID'] = df_sa_meta['PMID'].astype(str).str.strip()

df_black_abs['PMID'] = df_black_abs['PMID'].astype(str).str.strip()
df_sa_abs['PMID'] = df_sa_abs['PMID'].astype(str).str.strip()

In [63]:
df_black = pd.merge(df_black_meta, df_black_abs, on="PMID", how="inner")
df_sa = pd.merge(df_sa_meta, df_sa_abs, on="PMID", how="inner")

print("Black merged rows:", len(df_black))
print("South Asian merged rows:", len(df_sa))

Black merged rows: 0
South Asian merged rows: 0


In [64]:
import pandas as pd

def parse_pubmed_txt(file_path):
    pmid_list = []
    title_list = []
    abstract_list = []
    affiliation_list = []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    pmid = title = abstract = affiliation = ""
    current_field = None

    for line in lines:
        line = line.rstrip()

        if line.startswith("PMID-"):
            if pmid:
                pmid_list.append(pmid)
                title_list.append(title)
                abstract_list.append(abstract)
                affiliation_list.append(affiliation)

                title = abstract = affiliation = ""

            pmid = line.replace("PMID- ", "").strip()
            current_field = None

        elif line.startswith("TI  -"):
            title = line.replace("TI  - ", "").strip()
            current_field = "TI"

        elif line.startswith("AB  -"):
            abstract = line.replace("AB  - ", "").strip()
            current_field = "AB"

        elif line.startswith("AD  -"):
            affiliation = line.replace("AD  - ", "").strip()
            current_field = "AD"

        else:
            # Handle multi-line fields
            if current_field == "AB":
                abstract += " " + line.strip()
            elif current_field == "TI":
                title += " " + line.strip()
            elif current_field == "AD":
                affiliation += " " + line.strip()

    # Add last record
    if pmid:
        pmid_list.append(pmid)
        title_list.append(title)
        abstract_list.append(abstract)
        affiliation_list.append(affiliation)

    return pd.DataFrame({
        "PMID": pmid_list,
        "Title_txt": title_list,
        "Abstract": abstract_list,
        "Affiliation": affiliation_list
    })

In [65]:
# Load CSVs
df_black_csv = pd.read_csv("H2_black.csv")
df_sa_csv = pd.read_csv("H2_SA.csv")

# Parse TXTs
df_black_txt = parse_pubmed_txt("H2_black_pubmed_.txt")
df_sa_txt = parse_pubmed_txt("H2_SA_pubmed_.txt")

In [66]:
# Convert ALL PMIDs to clean strings
for df in [df_black_csv, df_sa_csv, df_black_txt, df_sa_txt]:
    df['PMID'] = df['PMID'].astype(str).str.strip()

In [67]:
df_black = pd.merge(df_black_csv, df_black_txt, on="PMID", how="inner")
df_sa = pd.merge(df_sa_csv, df_sa_txt, on="PMID", how="inner")

print("Black merged:", len(df_black))
print("SA merged:", len(df_sa))

Black merged: 3499
SA merged: 2218


In [68]:
df_black['Affiliation'] = df_black['Affiliation'].astype(str).str.lower()
df_sa['Affiliation'] = df_sa['Affiliation'].astype(str).str.lower()

us_keywords = ["united states", "usa", "u.s.", "us ", "american"]

df_black['flag_us'] = df_black['Affiliation'].str.contains('|'.join(us_keywords), na=False)
df_sa['flag_us'] = df_sa['Affiliation'].str.contains('|'.join(us_keywords), na=False)

df_black_us = df_black[df_black['flag_us']].copy()
df_sa_us = df_sa[df_sa['flag_us']].copy()

print("Black US:", len(df_black_us))
print("SA US:", len(df_sa_us))

Black US: 3399
SA US: 2096


In [69]:
df_black_us['Abstract'] = df_black_us['Abstract'].astype(str).str.lower()
df_sa_us['Abstract'] = df_sa_us['Abstract'].astype(str).str.lower()

risk_keywords = [
    "disproportionate", "higher risk", "elevated risk", "increased risk",
    "greater risk", "excess risk", "predisposition", "disparities",
    "unequal burden", "higher prevalence", "greater incidence",
    "more likely to", "susceptibility", "cardiovascular risk"
]

df_black_us['flag_risk'] = df_black_us['Abstract'].str.contains('|'.join(risk_keywords), na=False)
df_sa_us['flag_risk'] = df_sa_us['Abstract'].str.contains('|'.join(risk_keywords), na=False)

df_black_final = df_black_us[df_black_us['flag_risk']].copy()
df_sa_final = df_sa_us[df_sa_us['flag_risk']].copy()

print("Black final:", len(df_black_final))
print("SA final:", len(df_sa_final))

Black final: 1561
SA final: 771


In [70]:
df_black_final.to_csv("black_final.csv", index=False)
df_sa_final.to_csv("sa_final.csv", index=False)

In [2]:
import pandas as pd

# Load CSV
df = pd.read_csv("sa_final.csv")

# Fill empty Abstract or Affiliation cells
df['Abstract'] = df['Abstract'].fillna('')
df['Affiliation'] = df['Affiliation'].fillna('')

# Combine text for searching
df['text_to_search'] = df['Title'] + ' ' + df['Abstract'] + ' ' + df['Affiliation']

In [4]:
# US keywords
us_keywords = ["United States", "USA", "U.S.", "US population", "American"]

# Multi-site / national cohort keywords
multi_site_keywords = ["multi-site", "multi-center", "multicenter", "national survey", "cohort", "NHANES", "MASALA", "multistate", "multi-state"]

# Cardiovascular keywords
cvd_keywords = ["cardiovascular", "heart disease", "coronary", "myocardial infarction", "stroke", "CVD", "atherosclerosis"]

In [5]:
# Function to check if any keyword is in text
def contains_any(text, keywords):
    text = str(text).lower()
    return any(k.lower() in text for k in keywords)

# Apply filters
df_filtered = df[
    df['text_to_search'].apply(lambda x: contains_any(x, us_keywords)) &
    df['text_to_search'].apply(lambda x: contains_any(x, multi_site_keywords)) &
    df['text_to_search'].apply(lambda x: contains_any(x, cvd_keywords))
]

# Check results
print(f"Number of likely landmark South Asian studies: {len(df_filtered)}")

Number of likely landmark South Asian studies: 191


In [6]:
df_filtered = df_filtered.sort_values('Publication Year', ascending=True)

In [7]:
# Save filtered studies to a new CSV
df_filtered.to_csv("south_asian_landmark_candidates.csv", index=False)

print("Filtered CSV saved! You can open it now in Excel or Sheets.")

Filtered CSV saved! You can open it now in Excel or Sheets.


In [1]:
pip install biopython pandas

Note: you may need to restart the kernel to use updated packages.
